# 03 — Il confronto

Tutti i modelli finiti (feature, baseline, transformer con i loro seed), sugli stessi fold **per gruppo**. Le
metriche principali, fissate prima dei risultati: **F1 macro a 5 classi** e **accuratezza binaria**, per
testata. Si riportano anche accuratezza a 5 classi, tolerant 0.5 e 1, e il dettaglio per livello.

Gli intervalli al 95% vengono da un **bootstrap per gruppo** (2000 ricampionamenti dei gruppi di testate: le
testate dello stesso gruppo sbagliano insieme). Per ogni transformer, la **differenza** con il classificatore a
feature (quello con C scelto in validazione interna, `feature_tuned`) sugli stessi ricampionamenti.

Per le configurazioni con più seed c'è anche l'**ensemble** (`..._ensembleN`): la media delle probabilità dei seed
per ogni articolo, decisa prima di vedere i risultati.

Gira da solo alla fine di ogni esperimento della coda: si può guardare anche mentre la coda lavora.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

from comune import CLASSI, RESULTS, VERSIONI, metriche_di, per_testata, per_testata_media

SCHEMA = "fold_gruppo"
B = 2000

In [ ]:
modelli = {}
for f in sorted(RESULTS.glob(f"feature*_{SCHEMA}_articoli.csv")):
    modelli[f.name.replace(f"_{SCHEMA}_articoli.csv", "")] = pd.read_csv(f)
for d in sorted((RESULTS / "transformer").glob("*")):
    if (d / "FATTO").exists():
        modelli[d.name] = pd.read_csv(d / "articoli.csv")
# ensemble dei seed: per ogni configurazione con più seed finiti, la media delle probabilità per articolo
per_config = {}
for nome in list(modelli):
    m = re.match(r"(.+)_s\d+$", nome)
    if m:
        per_config.setdefault(m[1], []).append(nome)
for config, nomi_seed in per_config.items():
    if len(nomi_seed) > 1:
        parti = [modelli[n].set_index("article_id") for n in nomi_seed]
        e = parti[0][["domain", "fold", "y_true"]].copy()
        e[CLASSI] = sum(p[CLASSI] for p in parti) / len(parti)
        e["y_pred"] = e[CLASSI].to_numpy().argmax(1)
        modelli[f"{config}_ensemble{len(nomi_seed)}"] = e.reset_index()
print("modelli finiti:", list(modelli))
gruppo = pd.read_csv(VERSIONI / "fold_finale.csv").set_index("domain").gruppo

In [ ]:
PRINCIPALI = ["f1_macro", "bin_accuratezza", "accuratezza", "tolerant_0.5", "tolerant_1", "bin_f1_macro"]
righe, testate = {}, {}
for nome, p in modelli.items():
    for agg, fun in [("voto", per_testata), ("media", per_testata_media)]:
        t = fun(p); testate[(nome, agg)] = t
        righe[(nome, agg)] = {k: v for k, v in metriche_di(t.y_true, t.y_pred).items() if not isinstance(v, list)}
for f in sorted(RESULTS.glob(f"baseline_*_{SCHEMA}_metriche.json")):
    righe[(f.name.split(f"_{SCHEMA}")[0], "-")] = {k: v for k, v in json.load(open(f))["testata_voto"].items() if not isinstance(v, list)}
tutto = pd.DataFrame(righe).T
tutto.index.names = ["modello", "aggregazione"]
tutto.to_csv(RESULTS / "03_confronto_tutte_le_metriche.csv")
tutto[PRINCIPALI].round(3)

## Seed: media e deviazione standard

Gli esperimenti con lo stesso nome a parte il seed (`..._s42`, `..._s1`, `..._s2`).

In [ ]:
x = tutto[PRINCIPALI].reset_index()
x["config"] = x.modello.str.replace(r"_s\d+$", "", regex=True)
seed = x.groupby(["config", "aggregazione"])[PRINCIPALI].agg(["mean", "std", "count"])
seed.to_csv(RESULTS / "03_confronto_seed.csv")
seed.round(3)

## Intervalli e differenze con il feature (bootstrap per gruppo)

In [ ]:
rif = ("feature_tuned", "voto") if ("feature_tuned", "voto") in testate else ("feature", "voto")   # il feature al suo meglio, se c'è
nomi = [k for k in testate if k[1] == "media" or k == rif]
dom = testate[rif].index
g = gruppo.reindex(dom).to_numpy()
gruppi = np.unique(g)
pos = {x: np.where(g == x)[0] for x in gruppi}
rng = np.random.default_rng(0)
campioni = [np.concatenate([pos[x] for x in rng.choice(gruppi, len(gruppi))]) for _ in range(B)]


def due(t, idx):
    m = metriche_di(t.y_true.to_numpy()[idx], t.y_pred.to_numpy()[idx])
    return m["f1_macro"], m["bin_accuratezza"]


boot = {k: np.array([due(testate[k].reindex(dom), idx) for idx in campioni]) for k in nomi}
righe = {}
for k in nomi:
    b = boot[k]
    r = {"F1 macro": tutto.loc[k, "f1_macro"], "F1 2.5%": np.percentile(b[:, 0], 2.5), "F1 97.5%": np.percentile(b[:, 0], 97.5),
         "binario": tutto.loc[k, "bin_accuratezza"], "bin 2.5%": np.percentile(b[:, 1], 2.5), "bin 97.5%": np.percentile(b[:, 1], 97.5)}
    if k != rif:
        d = b - boot[rif]
        r.update({"ΔF1 vs feature": tutto.loc[k, "f1_macro"] - tutto.loc[rif, "f1_macro"], "ΔF1 2.5%": np.percentile(d[:, 0], 2.5),
                  "ΔF1 97.5%": np.percentile(d[:, 0], 97.5), "P(ΔF1>0)": (d[:, 0] > 0).mean(),
                  "Δbin vs feature": tutto.loc[k, "bin_accuratezza"] - tutto.loc[rif, "bin_accuratezza"],
                  "Δbin 2.5%": np.percentile(d[:, 1], 2.5), "Δbin 97.5%": np.percentile(d[:, 1], 97.5), "P(Δbin>0)": (d[:, 1] > 0).mean()})
    righe[f"{k[0]} ({k[1]})"] = r
intervalli = pd.DataFrame(righe).T
intervalli.to_csv(RESULTS / "03_confronto_intervalli.csv")
intervalli.round(3)

## Per livello e matrici di confusione (testata, media delle probabilità; feature col voto)

In [ ]:
LIV = ["l1", "l2", "l3", "l4", "l5"]
per_livello = pd.DataFrame({f"{k[0]} ({k[1]})": {f"{l} {q}": tutto.loc[k, f"{l}_{q}"] for l in LIV for q in ["recall", "f1"]} for k in nomi}).T
display(per_livello.round(2))
for k in nomi:
    t = testate[k]
    print(f"\n{k[0]} ({k[1]})")
    print(pd.crosstab(t.y_true.map(dict(enumerate(LIV))), t.y_pred.map(dict(enumerate(LIV))), rownames=["vero"], colnames=["predetto"]).to_string())